# 11a — Recursive Multi-Step Forecasting (skill-decay baseline)

Multi-step Arctic sea-ice-extent forecasting by **recursive / autoregressive rollout**: the model
predicts one day ahead, its own prediction is fed back in as the input for the next day, and errors
compound. It reuses the best **lagged multivariate LSTM** selected in `05_multivariate_lstm`.

**Role in the project.** This is the *recursive* companion to the planned **direct** multi-step
model `11b_extended_horizon_seq2seq` (experiment **E3**). Recursive rollout is the naive baseline
whose error accumulation the project's headline **skill-decay curve** is meant to improve on, so the
two belong to the same `11` family and should be read together.

**Conventions applied:**
- Evaluated against **persistence** and **climatology** baselines via `src.evaluation_utils`;
  headline results logged to `results/model_comparison.csv` (teacher forcing `scale="daily"`,
  recursive rollout and per-horizon results `scale="multi-day"`).
- Figures saved to `results/figures/` (prefixed `11a_`).
- No training happens here — the model is loaded with fixed weights already selected on `05`'s own
  validation era, so the two-way history/test split is intentional (no separate validation era needed).

> Status: **pending GPU run** — outputs cleared; requires the trained model from `05` and a full rerun.

In [ ]:
import pandas as pd
import numpy as np
import torch
import torch.nn as nn
import matplotlib.pyplot as plt
import seaborn as sns
from copy import deepcopy
import sys

sys.path.append('..')
from src.data_utils import load_data
from src.evaluation_utils import (
    PersistenceModel,
    ClimatologyModel,
    compute_all_metrics,
    compute_rmse,
    denormalize,
    log_model_results,
)

# Set style for better visualizations
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 6)

## 1. Load Data and Model Architecture

We'll use the **lagged multivariate LSTM** which showed the best performance in previous experiments.

In [ ]:
# This notebook EVALUATES a model already trained and selected in 05_multivariate_lstm — it loads
# fixed weights below, so no training or model selection happens here. A separate validation era is
# therefore unnecessary: `train_data` serves only as (a) the source of normalization statistics
# (which must match the trained model) and (b) the history for the day-of-year climatology baseline.
# The test era (2020–2023) is the untouched hold-out, consistent with the project split convention.
train_data = load_data(regions='pan_arctic', years=range(1989, 2020))
test_data  = load_data(regions='pan_arctic', years=range(2020, 2024))

print(f"History (train) data: {train_data.shape}")
print(f"Test data:            {test_data.shape}")
print(f"\nDate range: {train_data['date'].min()} to {test_data['date'].max()}")

In [ ]:
# Define features (same as best model from 05_multivariate_lstm.ipynb)
features = [
    'extent_mkm2',
    't2m_mean',
    't2m_std',
    'msl_mean',
    'msl_std',
    'wind_speed_mean',
    'wind_speed_std',
]

lag_features = {
    'extent_mkm2': [15, 30, 60],
    't2m_mean': [15, 30, 60],
}

print("Base features:", features)
print("Lag features:", lag_features)

## 2. Dataset Class with Autoregressive Support

We'll reuse the dataset class but add methods for autoregressive prediction.

In [ ]:
class MultivariateArcticDataset(torch.utils.data.Dataset):
    """Dataset for multivariate LSTM with support for lagged features."""
    
    def __init__(self, data, sequence_length=30, forecast_horizon=1, 
                 features=None, target='extent_mkm2', scaler=None, 
                 lag_features=None, add_cyclical_time=False):
        self.data = data.sort_values('date').reset_index(drop=True)
        self.sequence_length = sequence_length
        self.forecast_horizon = forecast_horizon
        self.target = target
        
        if features is None:
            self.features = ['extent_mkm2']
        else:
            self.features = features.copy()
        
        if add_cyclical_time:
            day_of_year = pd.to_datetime(self.data['date']).dt.dayofyear
            self.data['day_of_year_sin'] = np.sin(2 * np.pi * day_of_year / 365.25)
            self.data['day_of_year_cos'] = np.cos(2 * np.pi * day_of_year / 365.25)
            self.features.extend(['day_of_year_sin', 'day_of_year_cos'])
        
        # Store lag configuration for autoregressive prediction
        self.lag_features = lag_features
        
        if lag_features is not None:
            for column, lags in lag_features.items():
                for lag_days in lags:
                    lagged_column_name = f"{column}_lag{lag_days}"
                    self.data[lagged_column_name] = self.data[column].shift(lag_days)
                    self.features.append(lagged_column_name)
            
            self.data = self.data.dropna().reset_index(drop=True)
        
        self.data_values = self.data[self.features].values.astype(np.float32)
        self.target_idx = self.features.index(self.target)
        
        if scaler is None:
            self.mean = self.data_values.mean(axis=0, keepdims=True)
            self.std = self.data_values.std(axis=0, keepdims=True)
            self.std = np.where(self.std == 0, 1.0, self.std)
        else:
            self.mean, self.std = scaler
        
        self.data_normalized = (self.data_values - self.mean) / self.std
    
    def __len__(self):
        return len(self.data_normalized) - self.sequence_length - self.forecast_horizon + 1
    
    def __getitem__(self, idx):
        X = self.data_normalized[idx:idx + self.sequence_length]
        y = self.data_normalized[idx + self.sequence_length + self.forecast_horizon - 1][self.target_idx]
        
        X = torch.tensor(X, dtype=torch.float32)
        y = torch.tensor(y, dtype=torch.float32)
        
        return X, y
    
    def get_sequence_at_date(self, date_idx):
        """Get a sequence ending at a specific date index for autoregressive prediction."""
        if date_idx < self.sequence_length:
            raise ValueError(f"Not enough history for date_idx {date_idx}")
        
        X = self.data_normalized[date_idx - self.sequence_length:date_idx]
        return torch.tensor(X, dtype=torch.float32)

## 3. Model Architecture

Same LSTM architecture used in the multivariate notebook.

In [ ]:
class IceExtentLSTM(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, output_size=1, dropout=0.2):
        super(IceExtentLSTM, self).__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0
        )
        
        self.fc = nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        h_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        c_0 = torch.zeros(self.num_layers, x.size(0), self.hidden_size).to(x.device)
        
        out, (h_n, c_n) = self.lstm(x, (h_0, c_0))
        out = self.fc(out[:, -1, :])
        
        return out

## 4. Create Dataset and Load Best Model

In [ ]:
# Create datasets
train_dataset = MultivariateArcticDataset(
    train_data, 
    sequence_length=30, 
    forecast_horizon=1,
    features=features,
    target='extent_mkm2',
    lag_features=lag_features
)

test_dataset = MultivariateArcticDataset(
    test_data,
    sequence_length=30,
    forecast_horizon=1,
    features=features,
    target='extent_mkm2',
    scaler=(train_dataset.mean, train_dataset.std),
    lag_features=lag_features
)

print(f"Training samples: {len(train_dataset)}")
print(f"Test samples: {len(test_dataset)}")
print(f"Number of features: {len(train_dataset.features)}")
print(f"\nFeatures: {train_dataset.features}")

In [ ]:
# Load the best model
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Using device: {device}")

model = IceExtentLSTM(
    input_size=len(train_dataset.features),
    hidden_size=64,
    num_layers=2,
    output_size=1,
    dropout=0.2
)

# Load weights for the best lagged multivariate LSTM produced by 05_multivariate_lstm.
# Checkpoints live in ../models/ (PyTorch .pt files are git-ignored). Adjust the filename here if
# 05 saves the model under a different name.
model.load_state_dict(torch.load('../models/best_lagged_model.pt', map_location=device))
model = model.to(device)
model.eval()

print(f"Model loaded successfully!")
print(f"Parameters: {sum(p.numel() for p in model.parameters()):,}")

## 5. Teacher Forcing Evaluation (Baseline)

First, let's evaluate using teacher forcing (standard approach) to establish a baseline.

In [ ]:
def evaluate_teacher_forcing(model, dataset, device):
    """Standard evaluation using actual historical data at each step."""
    model.eval()
    
    predictions = []
    actuals = []
    dates = []
    
    with torch.no_grad():
        for i in range(len(dataset)):
            X, y = dataset[i]
            X = X.unsqueeze(0).to(device)  # Add batch dimension
            
            pred = model(X)
            predictions.append(pred.cpu().item())
            actuals.append(y.item())
            
            # Get corresponding date
            date_idx = i + dataset.sequence_length
            dates.append(dataset.data['date'].iloc[date_idx])
    
    return np.array(predictions), np.array(actuals), pd.Series(dates)

# Evaluate with teacher forcing
print("Evaluating with teacher forcing...")
tf_predictions, tf_actuals, tf_dates = evaluate_teacher_forcing(model, test_dataset, device)

print(f"Generated {len(tf_predictions)} predictions")
print(f"Date range: {tf_dates.iloc[0].strftime('%Y-%m-%d')} to {tf_dates.iloc[-1].strftime('%Y-%m-%d')}")

In [ ]:
# Denormalize predictions
train_mean = train_dataset.mean[0, train_dataset.target_idx]
train_std = train_dataset.std[0, train_dataset.target_idx]

tf_predictions_mkm2 = denormalize(tf_predictions, train_mean, train_std)
tf_actuals_mkm2 = denormalize(tf_actuals, train_mean, train_std)

# --- Baselines on the test era (denormalized, Mkm²) --------------------------------------
# Persistence: forecast(t) = observed extent at t-1. Teacher-forcing targets align to dataset rows
# starting at `sequence_length`, so persistence is the extent one row earlier. Climatology: the
# day-of-year mean fit on the history (train) era only.
seq = test_dataset.sequence_length
extent_phys = test_dataset.data['extent_mkm2'].values
n_tf = len(tf_predictions_mkm2)
tf_persistence = extent_phys[seq - 1 : seq - 1 + n_tf]

clim = ClimatologyModel()
clim.fit(dates=train_data['date'], values=train_data['extent_mkm2'])
tf_climatology = clim.predict(tf_dates)

# Compute metrics WITH baselines so skill scores are populated
tf_metrics = compute_all_metrics(
    y_true=tf_actuals_mkm2, y_pred=tf_predictions_mkm2,
    y_baseline_persistence=tf_persistence, y_baseline_climatology=tf_climatology,
    climatology=tf_climatology,
)

print("\n=== TEACHER FORCING PERFORMANCE (1-day, test era) ===")
print(f"RMSE: {tf_metrics['rmse']:.4f} Mkm²")
print(f"MAE:  {tf_metrics['mae']:.4f} Mkm²")
print(f"MAPE: {tf_metrics['mape']:.2f}%")
print(f"Skill vs persistence: {tf_metrics['skill_score_persistence']:.2%}")
print(f"Skill vs climatology: {tf_metrics['skill_score_climatology']:.2%}")

log_model_results(
    model_name="LSTM_lagged_TeacherForcing_1day",
    metrics=tf_metrics,
    scale="daily",
    metadata={"mode": "teacher_forcing", "lookback_days": int(seq),
              "features": "multivariate + lags", "test_period": "2020-2023"},
    output_file="../results/model_comparison.csv",
)

## 6. Autoregressive Prediction Implementation

Now we implement **true autoregressive prediction** where the model's predictions are fed back as inputs.

### Challenge with Lagged Features

Our model uses lagged features (extent_lag7, extent_lag14, extent_lag30, t2m_lag7, etc.). In autoregressive mode:
- We need to track which values are predictions vs actuals
- Update the sliding window correctly as we predict forward
- Maintain all features (weather variables + lags) in normalized space

In [ ]:
def autoregressive_predict(model, dataset, start_idx, num_steps, device):
    """
    Perform autoregressive prediction for multiple steps ahead.
    
    Args:
        model: Trained LSTM model
        dataset: MultivariateArcticDataset instance
        start_idx: Starting date index in the dataset
        num_steps: Number of days to predict ahead
        device: torch device
    
    Returns:
        predictions: Array of predictions (normalized)
        actuals: Array of actual values (normalized)
        dates: Dates for each prediction
    """
    model.eval()
    
    predictions = []
    actuals = []
    dates = []
    
    # Get initial sequence (all actual data)
    sequence_length = dataset.sequence_length
    
    # We need a mutable copy of the normalized data to update with predictions
    # Start from the beginning to have enough history for lags
    data_copy = dataset.data_normalized.copy()
    
    with torch.no_grad():
        for step in range(num_steps):
            current_idx = start_idx + step
            
            # Check bounds
            if current_idx >= len(dataset.data):
                break
            
            # Get the sequence for prediction
            X = data_copy[current_idx - sequence_length:current_idx]
            X = torch.tensor(X, dtype=torch.float32).unsqueeze(0).to(device)
            
            # Make prediction
            pred = model(X)
            pred_value = pred.cpu().item()
            
            # Get actual value
            actual_value = dataset.data_normalized[current_idx][dataset.target_idx]
            
            predictions.append(pred_value)
            actuals.append(actual_value)
            dates.append(dataset.data['date'].iloc[current_idx])
            
            # KEY STEP: Update data_copy with prediction for extent_mkm2
            # This prediction will be used in future timesteps via lags
            data_copy[current_idx, dataset.target_idx] = pred_value
            
            # Update lagged extent features
            # Note: Other features (t2m, msl, wind) remain as actuals since we don't predict them
            if dataset.lag_features is not None and 'extent_mkm2' in dataset.lag_features:
                for lag_days in dataset.lag_features['extent_mkm2']:
                    lag_feature_name = f'extent_mkm2_lag{lag_days}'
                    if lag_feature_name in dataset.features:
                        lag_feature_idx = dataset.features.index(lag_feature_name)
                        
                        # Get the value from lag_days ago (which might be a prediction)
                        source_idx = current_idx - lag_days
                        if source_idx >= 0:
                            data_copy[current_idx, lag_feature_idx] = data_copy[source_idx, dataset.target_idx]
    
    return np.array(predictions), np.array(actuals), pd.Series(dates)

## 7. Autoregressive Evaluation

Let's evaluate the full test set using autoregressive prediction.

In [ ]:
# Start from the first valid index (after sequence_length + max_lag)
start_idx = test_dataset.sequence_length
num_steps = len(test_dataset)

print(f"Starting autoregressive prediction from index {start_idx}")
print(f"Predicting {num_steps} steps ahead...")

ar_predictions, ar_actuals, ar_dates = autoregressive_predict(
    model, test_dataset, start_idx, num_steps, device
)

print(f"\nGenerated {len(ar_predictions)} predictions")
print(f"Date range: {ar_dates.iloc[0].strftime('%Y-%m-%d')} to {ar_dates.iloc[-1].strftime('%Y-%m-%d')}")

In [ ]:
# Denormalize predictions
ar_predictions_mkm2 = denormalize(ar_predictions, train_mean, train_std)
ar_actuals_mkm2 = denormalize(ar_actuals, train_mean, train_std)

# --- Baselines for the recursive rollout (denormalized, Mkm²) ----------------------------
# Same persistence / climatology references as teacher forcing, aligned to the rollout dates. NOTE:
# persistence uses the actual obs at t-1, which a recursive forecast does NOT have access to at
# deployment — so a strongly negative skill-vs-persistence here is expected and is precisely the
# error-accumulation story this notebook exists to document.
n_ar = len(ar_predictions_mkm2)
ar_persistence = extent_phys[seq - 1 : seq - 1 + n_ar]
ar_climatology = clim.predict(ar_dates)

ar_metrics = compute_all_metrics(
    y_true=ar_actuals_mkm2, y_pred=ar_predictions_mkm2,
    y_baseline_persistence=ar_persistence, y_baseline_climatology=ar_climatology,
    climatology=ar_climatology,
)

print("\n=== AUTOREGRESSIVE (RECURSIVE) PERFORMANCE — full test rollout ===")
print(f"RMSE: {ar_metrics['rmse']:.4f} Mkm²")
print(f"MAE:  {ar_metrics['mae']:.4f} Mkm²")
print(f"MAPE: {ar_metrics['mape']:.2f}%")
print(f"Skill vs persistence: {ar_metrics['skill_score_persistence']:.2%}")
print(f"Skill vs climatology: {ar_metrics['skill_score_climatology']:.2%}")

log_model_results(
    model_name="LSTM_lagged_Autoregressive_rollout",
    metrics=ar_metrics,
    scale="multi-day",
    metadata={"mode": "recursive_autoregressive", "lookback_days": int(seq),
              "rollout_days": int(n_ar), "features": "multivariate + lags", "test_period": "2020-2023"},
    output_file="../results/model_comparison.csv",
)

## 8. Teacher Forcing vs Autoregressive Comparison

In [ ]:
# Align the datasets (they might have different lengths)
min_len = min(len(tf_predictions_mkm2), len(ar_predictions_mkm2))

comparison_df = pd.DataFrame({
    'Mode': ['Teacher Forcing', 'Autoregressive'],
    'RMSE': [tf_metrics['rmse'], ar_metrics['rmse']],
    'MAE': [tf_metrics['mae'], ar_metrics['mae']],
    'MAPE': [tf_metrics['mape'], ar_metrics['mape']],
    'Samples': [len(tf_predictions_mkm2), len(ar_predictions_mkm2)]
})

print("\n" + "="*70)
print("TEACHER FORCING vs AUTOREGRESSIVE COMPARISON")
print("="*70)
print(comparison_df.to_string(index=False))
print("="*70)

# Compute degradation
rmse_degradation = ((ar_metrics['rmse'] - tf_metrics['rmse']) / tf_metrics['rmse']) * 100
mae_degradation = ((ar_metrics['mae'] - tf_metrics['mae']) / tf_metrics['mae']) * 100

print(f"\nPerformance Degradation:")
print(f"  RMSE increases by {rmse_degradation:.1f}%")
print(f"  MAE increases by {mae_degradation:.1f}%")

## 9. Error Propagation Analysis

Analyze how errors compound over time in autoregressive mode.

In [ ]:
# Compute cumulative error metrics
ar_errors = ar_predictions_mkm2 - ar_actuals_mkm2
ar_abs_errors = np.abs(ar_errors)
ar_squared_errors = ar_errors ** 2

# Rolling statistics (30-day window)
window_size = 30
rolling_rmse = np.sqrt(pd.Series(ar_squared_errors).rolling(window=window_size).mean())
rolling_mae = pd.Series(ar_abs_errors).rolling(window=window_size).mean()

# Plot error evolution
fig, axes = plt.subplots(3, 1, figsize=(14, 12))

# Plot 1: Predictions vs Actuals
axes[0].plot(ar_dates, ar_actuals_mkm2, label='Actual', linewidth=2, alpha=0.7)
axes[0].plot(ar_dates, ar_predictions_mkm2, label='Autoregressive Prediction', linewidth=1.5, alpha=0.8)
axes[0].set_ylabel('Ice Extent (Mkm²)', fontsize=12)
axes[0].set_title('Autoregressive Predictions vs Actuals', fontsize=14, fontweight='bold')
axes[0].legend(loc='best')
axes[0].grid(True, alpha=0.3)

# Plot 2: Raw errors over time
axes[1].plot(ar_dates, ar_errors, color='red', alpha=0.6, linewidth=1)
axes[1].axhline(y=0, color='black', linestyle='--', linewidth=1)
axes[1].fill_between(ar_dates, 0, ar_errors, where=(ar_errors > 0), 
                      color='red', alpha=0.3, label='Overestimate')
axes[1].fill_between(ar_dates, 0, ar_errors, where=(ar_errors < 0), 
                      color='blue', alpha=0.3, label='Underestimate')
axes[1].set_ylabel('Prediction Error (Mkm²)', fontsize=12)
axes[1].set_title('Error Over Time (Prediction - Actual)', fontsize=14, fontweight='bold')
axes[1].legend(loc='best')
axes[1].grid(True, alpha=0.3)

# Plot 3: Rolling error metrics
axes[2].plot(ar_dates, rolling_rmse, label=f'{window_size}-day Rolling RMSE', 
             linewidth=2, color='darkred')
axes[2].plot(ar_dates, rolling_mae, label=f'{window_size}-day Rolling MAE', 
             linewidth=2, color='darkblue')
axes[2].set_xlabel('Date', fontsize=12)
axes[2].set_ylabel('Error (Mkm²)', fontsize=12)
axes[2].set_title('Rolling Error Metrics (Error Propagation)', fontsize=14, fontweight='bold')
axes[2].legend(loc='best')
axes[2].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/11a_autoregressive_error_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Error propagation visualization saved!")

## 10. Multi-Step Ahead Forecasting

Test autoregressive prediction for specific forecast horizons: 7, 14, 30, 60, 90 days.

In [ ]:
def multi_horizon_forecast(model, dataset, start_dates, horizons, device):
    """
    Evaluate autoregressive forecasts at multiple horizons.
    
    Args:
        model: Trained model
        dataset: Dataset instance
        start_dates: List of starting date indices
        horizons: List of forecast horizons (days ahead)
        device: torch device
    
    Returns:
        results: Dict mapping horizons to metrics
    """
    results = {h: {'predictions': [], 'actuals': [], 'dates': []} for h in horizons}
    
    for start_idx in start_dates:
        # Generate forecast sequence
        max_horizon = max(horizons)
        preds, acts, dates = autoregressive_predict(model, dataset, start_idx, max_horizon, device)
        
        # Extract predictions at specific horizons
        for h in horizons:
            if h <= len(preds):
                results[h]['predictions'].append(preds[h-1])  # h-1 because 0-indexed
                results[h]['actuals'].append(acts[h-1])
                results[h]['dates'].append(dates.iloc[h-1])
    
    # Compute metrics for each horizon
    metrics_by_horizon = {}
    for h in horizons:
        preds_h = np.array(results[h]['predictions'])
        acts_h = np.array(results[h]['actuals'])
        
        # Denormalize
        preds_mkm2 = denormalize(preds_h, train_mean, train_std)
        acts_mkm2 = denormalize(acts_h, train_mean, train_std)
        
        metrics_by_horizon[h] = compute_all_metrics(y_true=acts_mkm2, y_pred=preds_mkm2)
    
    return metrics_by_horizon

In [ ]:
# Sample starting dates throughout the test period (every 30 days)
start_dates = list(range(test_dataset.sequence_length, len(test_dataset.data) - 100, 30))
horizons = [1, 7, 14, 30, 60, 90]

print(f"Evaluating {len(start_dates)} forecast starting points...")
print(f"Horizons: {horizons} days\n")

horizon_metrics = multi_horizon_forecast(model, test_dataset, start_dates, horizons, device)

# Display results
print("\n" + "="*80)
print("FORECAST PERFORMANCE BY HORIZON")
print("="*80)

horizon_df = pd.DataFrame([
    {
        'Horizon (days)': h,
        'RMSE (Mkm²)': metrics['rmse'],
        'MAE (Mkm²)': metrics['mae'],
        'MAPE (%)': metrics['mape'],
        'Samples': len(horizon_metrics[h])
    }
    for h, metrics in horizon_metrics.items()
])

print(horizon_df.to_string(index=False))
print("="*80)

# Log the skill-decay curve (RMSE/MAE/MAPE by horizon) to the shared comparison table so the
# recursive error-accumulation is visible alongside other models (scale='multi-day').
for h, m in horizon_metrics.items():
    log_model_results(
        model_name=f"LSTM_lagged_AR_{h}day",
        metrics=m,
        scale="multi-day",
        metadata={"mode": "recursive_autoregressive", "horizon_days": int(h),
                  "n_starts": len(start_dates), "test_period": "2020-2023"},
        output_file="../results/model_comparison.csv",
    )
print(f"\n✓ Logged {len(horizon_metrics)} horizon results to results/model_comparison.csv")

In [ ]:
# Visualize performance degradation with horizon
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

horizons_list = horizon_df['Horizon (days)'].values

# RMSE vs Horizon
axes[0].plot(horizons_list, horizon_df['RMSE (Mkm²)'], marker='o', linewidth=2, markersize=8)
axes[0].set_xlabel('Forecast Horizon (days)', fontsize=12)
axes[0].set_ylabel('RMSE (Mkm²)', fontsize=12)
axes[0].set_title('RMSE vs Forecast Horizon', fontsize=13, fontweight='bold')
axes[0].grid(True, alpha=0.3)

# MAE vs Horizon
axes[1].plot(horizons_list, horizon_df['MAE (Mkm²)'], marker='s', linewidth=2, 
             markersize=8, color='orange')
axes[1].set_xlabel('Forecast Horizon (days)', fontsize=12)
axes[1].set_ylabel('MAE (Mkm²)', fontsize=12)
axes[1].set_title('MAE vs Forecast Horizon', fontsize=13, fontweight='bold')
axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/11a_forecast_horizon_analysis.png', dpi=300, bbox_inches='tight')
plt.show()

print("Forecast horizon analysis saved!")

## 11. Seasonal Error Propagation

How does error propagation differ between winter (stable ice) and summer (rapid melt)?

In [ ]:
# Add season column
ar_months = ar_dates.dt.month
ar_seasons = ar_months.apply(lambda m: 'Winter' if m in [12, 1, 2, 3] else 
                                       'Summer' if m in [6, 7, 8, 9] else 'Transition')

# Compute metrics by season
seasonal_ar_results = {}

for season in ['Winter', 'Summer', 'Transition']:
    mask = ar_seasons == season
    if mask.sum() > 0:
        season_preds = ar_predictions_mkm2[mask]
        season_acts = ar_actuals_mkm2[mask]
        seasonal_ar_results[season] = compute_all_metrics(y_true=season_acts, y_pred=season_preds)

# Display results
print("\n" + "="*80)
print("AUTOREGRESSIVE PERFORMANCE BY SEASON")
print("="*80)

seasonal_df = pd.DataFrame([
    {
        'Season': season,
        'RMSE': metrics['rmse'],
        'MAE': metrics['mae'],
        'MAPE': metrics['mape'],
    }
    for season, metrics in seasonal_ar_results.items()
])

print(seasonal_df.to_string(index=False))
print("="*80)

## 12. Example: Long-Range Autoregressive Forecast

Visualize a single long-range forecast (90 days) to see error accumulation.

In [ ]:
# Pick a starting point in early 2020
example_start_date = pd.to_datetime('2020-06-01')
example_start_idx = test_dataset.data[test_dataset.data['date'] == example_start_date].index[0]

print(f"Generating 90-day forecast starting from {example_start_date.strftime('%Y-%m-%d')}...")

# Generate forecast
forecast_preds, forecast_acts, forecast_dates = autoregressive_predict(
    model, test_dataset, example_start_idx, 90, device
)

# Denormalize
forecast_preds_mkm2 = denormalize(forecast_preds, train_mean, train_std)
forecast_acts_mkm2 = denormalize(forecast_acts, train_mean, train_std)

# Plot
plt.figure(figsize=(14, 6))
plt.plot(forecast_dates, forecast_acts_mkm2, label='Actual', linewidth=3, color='black', alpha=0.7)
plt.plot(forecast_dates, forecast_preds_mkm2, label='Autoregressive Forecast', 
         linewidth=2, color='red', alpha=0.8, linestyle='--')

# Add shaded confidence regions based on cumulative error
errors = forecast_preds_mkm2 - forecast_acts_mkm2
cumulative_std = pd.Series(errors).expanding().std().values
plt.fill_between(forecast_dates, 
                 forecast_preds_mkm2 - cumulative_std, 
                 forecast_preds_mkm2 + cumulative_std,
                 alpha=0.2, color='red', label='±1 Cumulative Std')

plt.xlabel('Date', fontsize=12)
plt.ylabel('Ice Extent (Mkm²)', fontsize=12)
plt.title(f'90-Day Autoregressive Forecast from {example_start_date.strftime("%Y-%m-%d")}', 
          fontsize=14, fontweight='bold')
plt.legend(loc='best', fontsize=11)
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../results/figures/11a_long_range_forecast_example.png', dpi=300, bbox_inches='tight')
plt.show()

# Print metrics for this forecast
forecast_metrics = compute_all_metrics(y_true=forecast_acts_mkm2, y_pred=forecast_preds_mkm2)
print(f"\n90-Day Forecast Metrics:")
print(f"  RMSE: {forecast_metrics['rmse']:.4f} Mkm²")
print(f"  MAE:  {forecast_metrics['mae']:.4f} Mkm²")
print(f"  MAPE: {forecast_metrics['mape']:.2f}%")

## 13. Detailed 7-Day Forecast Analysis

Now let's focus specifically on **7-day forecasts** to understand short-term autoregressive behavior in detail. We'll examine 4 different starting dates across various seasons to see how the model performs under different conditions.

In [ ]:
# Set random seed for reproducibility
np.random.seed(42)

# Select 4 random start dates from different parts of the test period
# We'll stratify by season to get diverse conditions
test_data_sorted = test_dataset.data.copy()
test_data_sorted['month'] = pd.to_datetime(test_data_sorted['date']).dt.month

# Define season masks
winter_mask = test_data_sorted['month'].isin([12, 1, 2, 3])
spring_mask = test_data_sorted['month'].isin([4, 5])
summer_mask = test_data_sorted['month'].isin([6, 7, 8, 9])
fall_mask = test_data_sorted['month'].isin([10, 11])

# Get valid indices (need enough history and future data)
valid_start = test_dataset.sequence_length
valid_end = len(test_dataset.data) - 10  # Leave room for 7-day forecast

# Sample one date from each season
selected_indices = []
selected_dates = []
selected_seasons = []

for season_name, mask in [('Winter', winter_mask), ('Spring', spring_mask), 
                           ('Summer', summer_mask), ('Fall', fall_mask)]:
    # Get valid indices for this season
    season_indices = test_data_sorted[mask].index.values
    valid_season_indices = [i for i in season_indices if valid_start <= i < valid_end]
    
    if len(valid_season_indices) > 0:
        # Randomly select one
        selected_idx = np.random.choice(valid_season_indices)
        selected_indices.append(selected_idx)
        selected_dates.append(test_data_sorted.loc[selected_idx, 'date'])
        selected_seasons.append(season_name)

print("Selected 7-Day Forecast Starting Dates:\n")
for i, (date, season, idx) in enumerate(zip(selected_dates, selected_seasons, selected_indices)):
    extent = test_data_sorted.loc[idx, 'extent_mkm2']
    print(f"{i+1}. {date.strftime('%Y-%m-%d')} ({season}) - Extent: {extent:.2f} Mkm² - Index: {idx}")

### Generate 7-Day Forecasts for Each Starting Date

In [ ]:
# Generate forecasts for each selected date
forecast_horizon = 7
forecast_results = []

for idx, start_date, season in zip(selected_indices, selected_dates, selected_seasons):
    print(f"\nGenerating 7-day forecast from {start_date.strftime('%Y-%m-%d')} ({season})...")
    
    # Generate autoregressive forecast
    preds, acts, dates = autoregressive_predict(model, test_dataset, idx, forecast_horizon, device)
    
    # Denormalize
    preds_mkm2 = denormalize(preds, train_mean, train_std)
    acts_mkm2 = denormalize(acts, train_mean, train_std)
    
    # Calculate errors for each day
    errors = preds_mkm2 - acts_mkm2
    abs_errors = np.abs(errors)
    pct_errors = (errors / acts_mkm2) * 100
    
    # Store results
    forecast_results.append({
        'start_date': start_date,
        'season': season,
        'dates': dates,
        'predictions': preds_mkm2,
        'actuals': acts_mkm2,
        'errors': errors,
        'abs_errors': abs_errors,
        'pct_errors': pct_errors,
        'start_idx': idx
    })
    
    # Print summary
    rmse = np.sqrt(np.mean(errors**2))
    mae = np.mean(abs_errors)
    print(f"  RMSE: {rmse:.4f} Mkm²")
    print(f"  MAE:  {mae:.4f} Mkm²")
    print(f"  Max error: {abs_errors.max():.4f} Mkm² (Day {abs_errors.argmax() + 1})")

print("\n" + "="*80)
print(f"Generated {len(forecast_results)} forecasts successfully!")
print("="*80)

### Visualize Individual 7-Day Forecasts

Create detailed plots for each forecast showing predictions, actuals, and day-by-day errors.

In [ ]:
# Create a 2x2 grid of plots
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
axes = axes.flatten()

colors = ['#1f77b4', '#ff7f0e', '#2ca02c', '#d62728']

for i, result in enumerate(forecast_results):
    ax = axes[i]
    
    dates = result['dates']
    preds = result['predictions']
    acts = result['actuals']
    start_date = result['start_date']
    season = result['season']
    
    # Plot predictions and actuals
    ax.plot(range(1, 8), acts, 'o-', label='Actual', linewidth=2.5, 
            markersize=8, color='black', alpha=0.7)
    ax.plot(range(1, 8), preds, 's--', label='Autoregressive Prediction', 
            linewidth=2, markersize=7, color=colors[i], alpha=0.8)
    
    # Add error bars
    for day in range(7):
        ax.plot([day+1, day+1], [acts[day], preds[day]], 
                color='red', alpha=0.3, linewidth=1.5, linestyle=':')
    
    # Formatting
    ax.set_xlabel('Day Ahead', fontsize=11)
    ax.set_ylabel('Ice Extent (Mkm²)', fontsize=11)
    ax.set_title(f'{season}: {start_date.strftime("%Y-%m-%d")}\n' + 
                 f'RMSE: {np.sqrt(np.mean(result["errors"]**2)):.3f} Mkm²', 
                 fontsize=12, fontweight='bold')
    ax.legend(loc='best', fontsize=9)
    ax.grid(True, alpha=0.3)
    ax.set_xticks(range(1, 8))
    
    # Add extent range
    extent_range = acts.max() - acts.min()

plt.tight_layout()
plt.savefig('../results/figures/11a_7day_forecast_comparison.png', dpi=300, bbox_inches='tight')
plt.show()

print("Individual 7-day forecasts plotted!")

### Day-by-Day Error Progression Analysis

In [ ]:
# Analyze how errors evolve day-by-day across all forecasts
fig, axes = plt.subplots(2, 2, figsize=(16, 10))

# Plot 1: Absolute errors by day
ax1 = axes[0, 0]
for i, result in enumerate(forecast_results):
    ax1.plot(range(1, 8), result['abs_errors'], 'o-', 
             label=f"{result['season']}: {result['start_date'].strftime('%Y-%m-%d')}",
             linewidth=2, markersize=6, color=colors[i])
ax1.set_xlabel('Day Ahead', fontsize=11)
ax1.set_ylabel('Absolute Error (Mkm²)', fontsize=11)
ax1.set_title('Absolute Error Progression', fontsize=12, fontweight='bold')
ax1.legend(loc='best', fontsize=9)
ax1.grid(True, alpha=0.3)
ax1.set_xticks(range(1, 8))

# Plot 2: Signed errors (over/under estimation)
ax2 = axes[0, 1]
for i, result in enumerate(forecast_results):
    ax2.plot(range(1, 8), result['errors'], 'o-', 
             label=f"{result['season']}",
             linewidth=2, markersize=6, color=colors[i])
ax2.axhline(y=0, color='black', linestyle='--', linewidth=1, alpha=0.5)
ax2.set_xlabel('Day Ahead', fontsize=11)
ax2.set_ylabel('Signed Error (Mkm²)', fontsize=11)
ax2.set_title('Signed Error (Positive = Overestimate)', fontsize=12, fontweight='bold')
ax2.legend(loc='best', fontsize=9)
ax2.grid(True, alpha=0.3)
ax2.set_xticks(range(1, 8))

# Plot 3: Percentage errors
ax3 = axes[1, 0]
for i, result in enumerate(forecast_results):
    ax3.plot(range(1, 8), np.abs(result['pct_errors']), 'o-', 
             label=f"{result['season']}",
             linewidth=2, markersize=6, color=colors[i])
ax3.set_xlabel('Day Ahead', fontsize=11)
ax3.set_ylabel('Absolute Percentage Error (%)', fontsize=11)
ax3.set_title('Percentage Error Progression', fontsize=12, fontweight='bold')
ax3.legend(loc='best', fontsize=9)
ax3.grid(True, alpha=0.3)
ax3.set_xticks(range(1, 8))

# Plot 4: Cumulative error (expanding RMSE)
ax4 = axes[1, 1]
for i, result in enumerate(forecast_results):
    cumulative_rmse = [np.sqrt(np.mean(result['errors'][:day+1]**2)) 
                       for day in range(7)]
    ax4.plot(range(1, 8), cumulative_rmse, 'o-', 
             label=f"{result['season']}",
             linewidth=2, markersize=6, color=colors[i])
ax4.set_xlabel('Day Ahead', fontsize=11)
ax4.set_ylabel('Cumulative RMSE (Mkm²)', fontsize=11)
ax4.set_title('Cumulative RMSE Through Forecast', fontsize=12, fontweight='bold')
ax4.legend(loc='best', fontsize=9)
ax4.grid(True, alpha=0.3)
ax4.set_xticks(range(1, 8))

plt.tight_layout()
plt.savefig('../results/figures/11a_7day_error_progression.png', dpi=300, bbox_inches='tight')
plt.show()

print("Error progression analysis complete!")

### Comprehensive Comparison Table

In [ ]:
# Create detailed comparison table
print("\n" + "="*100)
print("7-DAY FORECAST DETAILED COMPARISON")
print("="*100)

for i, result in enumerate(forecast_results):
    print(f"\n{i+1}. {result['season'].upper()}: {result['start_date'].strftime('%Y-%m-%d')}")
    print("-" * 100)
    
    # Create day-by-day table
    day_by_day_df = pd.DataFrame({
        'Day': range(1, 8),
        'Date': [d.strftime('%Y-%m-%d') for d in result['dates']],
        'Actual (Mkm²)': result['actuals'],
        'Predicted (Mkm²)': result['predictions'],
        'Error (Mkm²)': result['errors'],
        'Abs Error (Mkm²)': result['abs_errors'],
        'Error (%)': result['pct_errors']
    })
    
    print(day_by_day_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))
    
    # Summary statistics
    print(f"\nSummary Statistics:")
    print(f"  RMSE:              {np.sqrt(np.mean(result['errors']**2)):.4f} Mkm²")
    print(f"  MAE:               {np.mean(result['abs_errors']):.4f} Mkm²")
    print(f"  Max Abs Error:     {result['abs_errors'].max():.4f} Mkm² (Day {result['abs_errors'].argmax() + 1})")
    print(f"  Mean % Error:      {np.mean(np.abs(result['pct_errors'])):.2f}%")
    print(f"  Initial Extent:    {result['actuals'][0]:.4f} Mkm²")
    print(f"  Final Extent:      {result['actuals'][-1]:.4f} Mkm²")
    print(f"  Extent Change:     {result['actuals'][-1] - result['actuals'][0]:.4f} Mkm²")

print("\n" + "="*100)

### Statistical Summary Across All 7-Day Forecasts

In [ ]:
# Aggregate statistics across all forecasts
print("\n" + "="*80)
print("AGGREGATE STATISTICS FOR 7-DAY FORECASTS")
print("="*80)

# Compute day-by-day average errors
day_errors = {day: [] for day in range(1, 8)}
day_abs_errors = {day: [] for day in range(1, 8)}
day_pct_errors = {day: [] for day in range(1, 8)}

for result in forecast_results:
    for day in range(7):
        day_errors[day+1].append(result['errors'][day])
        day_abs_errors[day+1].append(result['abs_errors'][day])
        day_pct_errors[day+1].append(np.abs(result['pct_errors'][day]))

# Create summary table
day_summary = pd.DataFrame({
    'Day': range(1, 8),
    'Mean Abs Error (Mkm²)': [np.mean(day_abs_errors[d]) for d in range(1, 8)],
    'Std Abs Error (Mkm²)': [np.std(day_abs_errors[d]) for d in range(1, 8)],
    'Mean % Error': [np.mean(day_pct_errors[d]) for d in range(1, 8)],
    'Max Abs Error (Mkm²)': [np.max(day_abs_errors[d]) for d in range(1, 8)],
    'Min Abs Error (Mkm²)': [np.min(day_abs_errors[d]) for d in range(1, 8)]
})

print("\nDay-by-Day Average Performance:")
print(day_summary.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

# Overall statistics
all_errors = np.concatenate([result['errors'] for result in forecast_results])
all_abs_errors = np.concatenate([result['abs_errors'] for result in forecast_results])
all_pct_errors = np.concatenate([np.abs(result['pct_errors']) for result in forecast_results])

print(f"\nOverall 7-Day Forecast Statistics (across {len(forecast_results)} forecasts):")
print(f"  RMSE:           {np.sqrt(np.mean(all_errors**2)):.4f} Mkm²")
print(f"  MAE:            {np.mean(all_abs_errors):.4f} Mkm²")
print(f"  Mean % Error:   {np.mean(all_pct_errors):.2f}%")
print(f"  Median % Error: {np.median(all_pct_errors):.2f}%")
print(f"  90th %ile Error: {np.percentile(all_abs_errors, 90):.4f} Mkm²")

print("\n" + "="*80)

### Key Insights from 7-Day Forecasts

#### Error Growth Pattern
- **Day 1**: Errors are minimal (similar to teacher forcing)
- **Days 2-3**: Errors begin to accumulate as lagged features use predictions
- **Days 4-7**: Error propagation becomes more pronounced

#### Seasonal Differences
- **Winter**: More stable, slower error growth (ice extent changes slowly)
- **Summer**: Faster error growth (rapid melt season with high variability)
- **Transition**: Moderate behavior between extremes

#### Practical Implications
- 7-day forecasts show reasonable accuracy for operational use
- Error propagation is manageable at this horizon
- Regular data updates (e.g., daily) can reset error accumulation
- Uncertainty should be communicated (expand with forecast horizon)

## 14. Key Findings Summary

In [ ]:
print("\n" + "="*80)
print("KEY FINDINGS: AUTOREGRESSIVE PREDICTION ANALYSIS")
print("="*80)

print("\n1. TEACHER FORCING vs AUTOREGRESSIVE GAP")
print(f"   - Teacher Forcing RMSE: {tf_metrics['rmse']:.4f} Mkm²")
print(f"   - Autoregressive RMSE:  {ar_metrics['rmse']:.4f} Mkm²")
print(f"   - Performance degradation: {rmse_degradation:.1f}%")
print(f"   → Real-world deployment will experience {rmse_degradation:.1f}% higher errors")

print("\n2. ERROR PROPAGATION BY HORIZON")
rmse_1day = horizon_metrics[1]['rmse']
rmse_30day = horizon_metrics[30]['rmse']
rmse_90day = horizon_metrics[90]['rmse']
print(f"   - 1-day ahead:  RMSE = {rmse_1day:.4f} Mkm²")
print(f"   - 30-day ahead: RMSE = {rmse_30day:.4f} Mkm² ({(rmse_30day/rmse_1day-1)*100:.1f}% increase)")
print(f"   - 90-day ahead: RMSE = {rmse_90day:.4f} Mkm² ({(rmse_90day/rmse_1day-1)*100:.1f}% increase)")
print(f"   → Errors grow approximately {(rmse_30day/rmse_1day-1)*100/30:.2f}% per day")

print("\n3. SEASONAL DIFFERENCES")
if 'Winter' in seasonal_ar_results and 'Summer' in seasonal_ar_results:
    winter_rmse = seasonal_ar_results['Winter']['rmse']
    summer_rmse = seasonal_ar_results['Summer']['rmse']
    print(f"   - Winter RMSE: {winter_rmse:.4f} Mkm²")
    print(f"   - Summer RMSE: {summer_rmse:.4f} Mkm²")
    if winter_rmse < summer_rmse:
        print(f"   → Winter forecasts are {((summer_rmse/winter_rmse-1)*100):.1f}% more accurate")
    else:
        print(f"   → Summer forecasts are {((winter_rmse/summer_rmse-1)*100):.1f}% more accurate")

print("\n4. PRACTICAL IMPLICATIONS")
print("   - The model shows significant error propagation in autoregressive mode")
print("   - Short-term forecasts (1-7 days) maintain good accuracy")
print("   - Long-term forecasts (30+ days) accumulate substantial errors")
print("   - Consider using teacher forcing with actual data when available")
print("   - For operational deployment, regularly update with observed data")

print("\n" + "="*80)